In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# P2-KN03 setup: repo, pinned packages (P2-KN01's set + google-genai as on the laptop), GPU check
!git clone -q https://github.com/amanchaurasia512/Self_optimized_Agent_Ai_chunking.git /kaggle/working/repo
!cd /kaggle/working/repo && git log --oneline -1
!pip install -q docling==2.143.0 jsonschema langchain-text-splitters==1.1.3 langchain-experimental==0.4.2 langchain-community==0.4.2 sentence-transformers==6.2.0 transformers==5.22.1 stanza==1.15.0 google-genai==2.34.0
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Copy only the tuning extractions from P2-KD01 into the repo (held-out files are never copied),
# and store the Gemini key from Kaggle Secrets where src/chunk/llm.py reads it (outside /kaggle/working, never printed).
import shutil
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

REPO = Path("/kaggle/working/repo")
sys.path.insert(0, str(REPO))
from src.common.split import tuning_ids

tuning = tuning_ids()
for parser in ["docling", "pdfplumber"]:
    target = REPO / "data" / "extracted" / parser
    target.mkdir(parents=True, exist_ok=True)
    sources = [path for path in Path("/kaggle/input").rglob(f"{parser}/*.json") if path.stem in tuning]
    for path in sources:
        shutil.copyfile(path, target / path.name)
    print(parser, ":", len(sources), "of", len(tuning), "tuning extractions copied")

key_file = Path.home() / ".config" / "gemini" / "api_key"
key_file.parent.mkdir(parents=True, exist_ok=True)
key_file.write_text(UserSecretsClient().get_secret("GEMINI_API_KEY").strip())
key_file.chmod(0o600)
print("Gemini key stored:", key_file.exists())

In [ ]:
# Every baseline setting (the same names as data/chunks on the laptop), on all tuning documents x both parsers.
import subprocess

SETTINGS = [("recursive", "size=512 overlap=8"), ("recursive", "size=512 overlap=32"), ("recursive", "size=512 overlap=64")]
failed = []
for chunker, settings in SETTINGS:
    command = f"python -m scripts.p2_chunk --chunker {chunker} {'--set ' + settings if settings else ''} --machine kaggle --run-source P2-KN03"
    result = subprocess.run(command, shell=True, cwd=REPO, capture_output=True, text=True)
    lines = (result.stdout + result.stderr).strip().splitlines()
    print(f"=== {chunker} {settings} | exit {result.returncode}\n" + "\n".join(lines[:1] + lines[-4:]) + "\n")
    if result.returncode:
        failed.append(f"{chunker} {settings}")
print("failed:", failed or "none")